# Proyek Akhir: Menyelesaikan Permasalahan Perusahaan Edutech

## Business Understanding
Jaya Jaya Institut ingin memahami faktor yang berkaitan dengan siswa yang tidak menyelesaikan studi dan memprioritaskan dukungan. Analisis ini menjawab komposisi status akhir, hubungan indikator akademik/administratif dengan status, dan kelayakan model baseline untuk menandai risiko dropout.

### Tujuan dan batasan
Target operasional notebook adalah klasifikasi biner `Dropout` (1) versus `Graduate` (0). Siswa berstatus `Enrolled` tidak digunakan untuk training maupun evaluasi; mereka disimpan terpisah sebagai data yang dapat diberi prediksi model. Data mencakup hasil hingga semester kedua sehingga model ini bukan prediksi dini sebelum semester berjalan selesai. Korelasi dan feature importance tidak membuktikan sebab-akibat.

- Nama: Gregorius Christian Sunaryo
- Email: christian10rks@gmail.com
- Id Dicoding: 

## Persiapan

### Menyiapkan library yang dibutuhkan

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

### Menyiapkan data yang akan digunakan

In [ ]:
student_performance_df = pd.read_csv(
    "https://raw.githubusercontent.com/dicodingacademy/dicoding_dataset/main/students_performance/data.csv",
    sep=";",
    encoding="utf-8-sig"
)

student_performance_df.head()

## Data Understanding

In [ ]:
# Struktur dataset
rows, cols = student_performance_df.shape

print(f"Ukuran dataset: {rows:,} baris × {cols} kolom")
student_performance_df.info()

In [ ]:
# Kualitas data
missing_values = student_performance_df.isna().sum()
duplicates = student_performance_df.duplicated().sum()

print(f"\nTotal nilai kosong  : {missing_values.sum():,}")
print(f"Jumlah baris duplikat: {duplicates:,}")

display(
    missing_values.sort_values(ascending=False)
    .head(10)
    .to_frame("Nilai Kosong")
)

In [ ]:
# Distribusi status mahasiswa
status_order = ["Dropout", "Enrolled", "Graduate"]

status_distribution = (
    student_performance_df["Status"]
    .value_counts()
    .reindex(status_order, fill_value=0)
    .rename_axis("Status")
    .to_frame("Jumlah")
)

status_distribution["Proporsi"] = (
    status_distribution["Jumlah"] / rows
)

display(
    status_distribution.style.format({
        "Jumlah": "{:,.0f}",
        "Proporsi": "{:.1%}",
    })
)

In [ ]:
# Visualisasi distribusi status
sns.countplot(
    data=student_performance_df,
    x="Status",
    order=status_order,
    color="#3973ac",
)

plt.title("Distribusi Status Akhir Mahasiswa")
plt.xlabel("Status")
plt.ylabel("Jumlah Mahasiswa")
plt.tight_layout()
plt.show()

In [ ]:
# Rata-rata capaian akademik berdasarkan status
academic_features = [
    "Curricular_units_1st_sem_approved",
    "Curricular_units_2nd_sem_approved",
    "Curricular_units_1st_sem_grade",
    "Curricular_units_2nd_sem_grade",
]

academic_summary = (
    student_performance_df
    .groupby("Status")[academic_features]
    .mean()
    .reindex(status_order)
    .round(2)
)

display(academic_summary)


## Data Preparation / Preprocessing
Model hanya menggunakan siswa berstatus `Dropout` dan `Graduate`. Target biner dipetakan secara eksplisit menjadi `Dropout = 1` dan `Graduate = 0`. Siswa `Enrolled` dikeluarkan dari training serta evaluasi dan disimpan terpisah sebagai data tanpa label untuk prediksi setelah model dilatih. Observasi dengan nilai kosong pada fitur terpilih dihapus.

In [ ]:
# Konfigurasi target dan fitur
status_column = "Status"
features = [
    "Age_at_enrollment",
    "Tuition_fees_up_to_date",
    "Scholarship_holder",
    "Debtor",
    "Curricular_units_1st_sem_enrolled",
    "Curricular_units_1st_sem_approved",
    "Curricular_units_1st_sem_grade",
    "Curricular_units_2nd_sem_enrolled",
    "Curricular_units_2nd_sem_approved",
    "Curricular_units_2nd_sem_grade",
]

# Pisahkan siswa berlabel dari siswa yang belum memiliki hasil akhir
labeled_data = student_performance_df[
    student_performance_df[status_column].isin(["Dropout", "Graduate"])
][features + [status_column]].dropna().copy()
enrolled_data = student_performance_df[
    student_performance_df[status_column].eq("Enrolled")
][features].dropna().copy()

X = labeled_data[features]
y = labeled_data[status_column].map({"Graduate": 0, "Dropout": 1}).astype(int)
X_enrolled = enrolled_data[features]

assert set(y.unique()) == {0, 1}
assert student_performance_df.loc[labeled_data.index, status_column].isin(
    ["Dropout", "Graduate"]
).all()

# Pertahankan proporsi target pada data latih dan uji
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)
print(f"Siswa untuk pemodelan: {len(labeled_data):,}")
print(f"Data latih: {len(X_train):,} | data uji: {len(X_test):,}")
print(f"Siswa Enrolled terpisah untuk prediksi: {len(X_enrolled):,}")
print("Distribusi target (0=Graduate, 1=Dropout):")
print(y.value_counts().sort_index())

## Modeling
Random Forest dilatih dan dievaluasi hanya pada siswa berstatus `Dropout` atau `Graduate`. Baris `Enrolled` tetap terpisah dan tidak ikut dalam proses training maupun evaluasi.

In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_probability = model.predict_proba(X_test)[:, 1]

print(f"Akurasi holdout: {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["Graduate", "Dropout"],
    digits=3,
    zero_division=0,
))
print("Confusion matrix [actual rows, predicted columns], labels [Graduate, Dropout]:")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

## Evaluation
Evaluasi holdout stratified hanya membandingkan kelas `Graduate` dan `Dropout`. Perhatikan precision, recall, dan F1 untuk kelas Dropout karena false negative berarti siswa berisiko tidak teridentifikasi.

In [ ]:
# Classification report
evaluation = pd.DataFrame(
    classification_report(
        y_test,
        y_pred,
        labels=[0, 1],
        target_names=["Graduate", "Dropout"],
        output_dict=True,
        zero_division=0,
    )
).T

print("Classification Report (0=Graduate, 1=Dropout):")
display(evaluation.round(3))

In [ ]:
# Confusion matrix
class_labels = ["Graduate", "Dropout"]

sns.heatmap(
    confusion_matrix(y_test, y_pred, labels=[0, 1]),
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_labels,
    yticklabels=class_labels,
)

plt.title("Confusion Matrix - Test Set")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
importance = (
    pd.Series(model.feature_importances_, index=features)
    .sort_values(ascending=False)
)

importance.sort_values().plot(
    kind="barh",
    figsize=(8, 5),
    color="#3973ac",
)

plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


In [ ]:
# Latih model final pada seluruh data berlabel untuk digunakan aplikasi
final_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
final_model.fit(X, y)

# Simpan artefak di folder model pada root proyek
project_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "app.py").is_file()),
    Path.cwd(),
)
model_path = project_root / "model" / "model.joblib"
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(final_model, model_path, compress=3)

# Verifikasi bahwa artefak dapat dimuat dan menghasilkan prediksi yang sama
loaded_model = joblib.load(model_path)
sample = X.head(10)
assert (final_model.predict(sample) == loaded_model.predict(sample)).all()
print(f"Model tersimpan: {model_path}")
print(f"Training rows (Dropout/Graduate): {len(X):,}")
print(f"Ukuran artefak: {model_path.stat().st_size:,} bytes")

In [ ]:
# Gunakan model hanya untuk memberi estimasi pada siswa Enrolled yang disisihkan
if not X_enrolled.empty:
    enrolled_predictions = pd.DataFrame(
        {
            "Predicted_status": pd.Series(
                final_model.predict(X_enrolled), index=X_enrolled.index
            ).map({0: "Graduate", 1: "Dropout"}),
            "Dropout_probability": final_model.predict_proba(X_enrolled)[:, 1],
        },
        index=X_enrolled.index,
    )
    print("Prediksi status untuk data Enrolled:")
    display(enrolled_predictions.head())
    print("Jumlah hasil prediksi:")
    display(enrolled_predictions["Predicted_status"].value_counts().to_frame("Jumlah"))
else:
    enrolled_predictions = pd.DataFrame(
        columns=["Predicted_status", "Dropout_probability"]
    )
    print("Tidak ada baris Enrolled dengan fitur lengkap untuk diprediksi.")

## Conclusion dan Action Items
Pada data historis, capaian unit dan nilai semester berkaitan dengan status akhir siswa, sementara status tunggakan dan pembayaran juga membantu membedakan profil risiko. Gunakan hasil dashboard dan prediksi untuk memprioritaskan percakapan dukungan, bukan untuk menjatuhkan sanksi atau membuat keputusan otomatis. Karena fitur model memuat informasi hingga semester kedua, model ini belum dapat diklaim sebagai peringatan dini.

1. Tawarkan konsultasi akademik ketika jumlah unit lulus atau nilai siswa menurun.
2. Arahkan siswa dengan kendala pembayaran ke dukungan finansial dan evaluasi kebutuhan bantuan.
3. Uji model pada kohort berikutnya, pantau recall/precision dan kalibrasi, serta audit performa lintas kelompok.
4. Kumpulkan snapshot semester dengan ID pseudonim agar tren siswa dan dampak intervensi dapat dievaluasi.

In [ ]:
# Top 10 features
print("Top 10 Feature Importance:")
display(importance.head(10).to_frame("Importance").round(4))